# Dev Inference Check — R-PEARL + LLM

Loads the checkpoint, verifies forward pass, then runs generation on one real sample.

**Use the `GREP-PRISM` conda env. All paths are relative to `notebooks/`.**

In [1]:
import sys, json
from ast import literal_eval

import torch
import torch_geometric.utils as pyg_utils

sys.path.insert(0, "../src")
sys.path.insert(0, "../SPINE/src")

from prism.models.loaders import graph_augmented_llm_from_pretrained
from prism.models.inference import GraphAugmentedInMemoryLLM
from prism.data.utils import safe_parse_graph

CKPT = "../checkpoints/dev_e1_rpearl_edges_eval/dev_e1_rpearl_edges_eval_rpearl_llm_r16"
DATA = "../data/gen/spine_exp1/formatted.json"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {DEVICE}")

/home/jporras/miniconda3/envs/GREP-PRISM/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: cuda


## 1 — Load model

In [2]:
model, tokenizer = graph_augmented_llm_from_pretrained(CKPT)
model.eval()
print(model)

Loading weights: 100%|██████████| 254/254 [00:00<00:00, 258.34it/s, Materializing param=model.norm.weight]                              
Skipping import of cpp extensions due to incompatible torch version 2.10.0+cu128 for torchao version 0.14.1             Please see https://github.com/pytorch/ao/issues/2919 for more info
/home/jporras/miniconda3/envs/GREP-PRISM/lib/python3.10/site-packages/peft/peft_model.py:598: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.we

GraphAugmentedLLM(
  (llm): PeftModelForCausalLM(
    (base_model): LoraModel(
      (model): LlamaForCausalLM(
        (model): LlamaModel(
          (embed_tokens): Embedding(128256, 3072)
          (layers): ModuleList(
            (0-27): 28 x LlamaDecoderLayer(
              (self_attn): LlamaAttention(
                (q_proj): lora.Linear(
                  (base_layer): Linear(in_features=3072, out_features=3072, bias=False)
                  (lora_dropout): ModuleDict(
                    (default): Dropout(p=0.2, inplace=False)
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=3072, out_features=16, bias=False)
                  )
                  (lora_B): ModuleDict(
                    (default): Linear(in_features=16, out_features=3072, bias=False)
                  )
                  (lora_embedding_A): ParameterDict()
                  (lora_embedding_B): ParameterDict()
                  (lora_magnitude_vect

## 2 — Parse one sample from training data

In [3]:
import re

with open(DATA) as f:
    samples = json.load(f)

sample = samples[0]
conversations = sample["conversations"]

# Extract scene graph from first user turn
user_content = conversations[0]["content"]
match = re.search(r"[Ss]cene graph:? ?(\{.*\})", user_content)
assert match, f"No scene graph found in: {user_content[:200]}"
scene_graph_dict = literal_eval(match.group(1))

print("Scene graph keys:", list(scene_graph_dict.keys()))
print("Nodes:", [n["name"] for n in scene_graph_dict.get("objects", []) + scene_graph_dict.get("regions", [])])

Scene graph keys: ['objects', 'regions', 'object_connections', 'region_connections', 'robot_location']
Nodes: ['house_1', 'house_2', 'grocery_store_1', 'shed_1', 'shed_1', 'example_road_1', 'example_road_2', 'field_11', 'field_13']


In [4]:
# Build PyG graph
nx_graph, _ = safe_parse_graph(scene_graph_dict)
node_names = list(nx_graph.nodes)
coords = torch.tensor([nx_graph.nodes[n]["coords"] for n in node_names], dtype=torch.float32)

pyg_graph = pyg_utils.from_networkx(nx_graph)
pyg_graph.coords = coords
pyg_graph.x = torch.zeros((coords.size(0), 1), dtype=torch.float32)
pyg_graph.node_names = node_names
pyg_graph.node_types = [nx_graph.nodes[n]["type"] for n in node_names]
pyg_graph.robot_location = scene_graph_dict.get("robot_location")

print(f"Nodes: {pyg_graph.num_nodes}, Edges: {pyg_graph.num_edges}")
print(f"node_names: {pyg_graph.node_names}")

Nodes: 8, Edges: 12
node_names: ['house_1', 'house_2', 'grocery_store_1', 'shed_1', 'example_road_1', 'example_road_2', 'field_11', 'field_13']


## 3 — Forward pass check

Tokenise the full conversation (user + assistant), run through the model, check that loss is a finite scalar and logits shape is correct.

In [5]:
# Tokenize full conversation (first two turns: user + first assistant reply)
turns = conversations[:2]  # [user, assistant]
messages = [{"role": m["role"], "content": m["content"]} for m in turns]

input_ids = tokenizer.apply_chat_template(
    messages, tokenize=True, return_tensors="pt"
)["input_ids"].to(DEVICE)

labels = input_ids.clone()
attention_mask = torch.ones_like(input_ids)

print(f"input_ids shape: {input_ids.shape}")

with torch.no_grad():
    out = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels,
        graphs=[pyg_graph],
    )

print(f"loss       : {out.loss.item():.4f}")
print(f"logits     : {out.logits.shape}")
assert torch.isfinite(out.loss), "Loss is not finite!"
assert out.logits.shape == (*input_ids.shape, model.config.vocab_size)
print("Forward pass OK")

input_ids shape: torch.Size([1, 469])
loss       : 6.1604
logits     : torch.Size([1, 469, 128256])
Forward pass OK


## 4 — Generation check

Use `GraphAugmentedInMemoryLLM` (the same class used in eval) on the first user turn.

In [ ]:
msg

[{'role': 'user',
  'content': "task: I need a shovel. Is there one in the scene?Scene graph:{'objects': [{'name': 'house_1', 'coords': [-1, -1]}, {'name': 'house_2', 'coords': [-3, -1]}, {'name': 'grocery_store_1', 'coords': [-5, -1]}, {'name': 'shed_1', 'coords': [1, 3]}, {'name': 'shed_1', 'coords': [1, 5]}], 'regions': [{'name': 'example_road_1', 'coords': [-1, 0]}, {'name': 'example_road_2', 'coords': [-2, 0]}, {'name': 'field_11', 'coords': [0, 1]}, {'name': 'field_13', 'coords': [2, 3]}], 'object_connections': [['house_1', 'example_road_1'], ['house_2', 'example_road_2'], ['shed_1', 'field_11'], ['shed_2', 'field_13']], 'region_connections': [['example_road_1', 'example_road_2'], ['example_road_1', 'field_11'], ['field_11', 'field_13']], 'robot_location': 'example_road_1'}"}]

: 

In [8]:
client = GraphAugmentedInMemoryLLM(model=model, tokenizer=tokenizer, device=DEVICE)

# Build the msg list the same way the eval loop does
msg = [{"role": "user", "content": user_content}]

response, ok = client.query_llm(msg,max_new_tokens=250)
print(f"query_llm returned ok={ok}")
print("\n--- Raw response ---")
print(response)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


query_llm returned ok=True

--- Raw response ---
'region':'region connections': [['region connections': [['region connections': [['region connections': [['region connections': [['region connections': [['region connections': [['region connections': [['region connections': [['region connections': [['region connections': [['region connections': [['region connections': [['region connections': [['region connections': [['region connections': [['region connections': [['region connections': [['region connections': [['region connections': [['region connections': [['region connections': [['region connections': [['region connections': [['region connections': [['region connections': [['region connections': [['region connections': [['region connections': [['region connections': [['region connections': [['region connections': [['region connections': [['region connections': [['region connections': [['region connections': [['region connections': [['region connections': [['region connections': [['regio

In [ ]:
# Attempt to parse as JSON (the model should produce valid JSON)
parsed = json.loads(response)
print("\n--- Parsed response ---")
for k, v in parsed.items():
    print(f"  {k}: {v}")

## 5 — Plain LLM baseline inference

Same prompt, same sample — but loaded as a vanilla `AutoModelForCausalLM` (no GNN head, no graph injection). Useful to sanity-check that the R-PEARL model is actually doing something different.

In [11]:
from prism.models.loaders import from_pretrained

with open(f"{CKPT}/gnn_config.json") as f:
    BASE_MODEL = json.load(f)["base_model"]
print(f"Base model: {BASE_MODEL}")

llm_baseline, llm_tokenizer = from_pretrained(BASE_MODEL)
llm_baseline.eval()
print("Plain LLM loaded.")

Base model: meta-llama/Llama-3.2-3B-Instruct


Loading weights: 100%|██████████| 254/254 [00:01<00:00, 236.80it/s, Materializing param=model.norm.weight]                              


Plain LLM loaded.


In [12]:
msg = [{"role": "user", "content": user_content}]

input_ids = llm_tokenizer.apply_chat_template(
    msg, tokenize=True, add_generation_prompt=True, return_tensors="pt"
)["input_ids"].to(DEVICE)

with torch.no_grad():
    outputs = llm_baseline.generate(
        input_ids=input_ids,
        max_new_tokens=4048, use_cache=True, temperature=0.01, min_p=0.1,
    )

out = llm_tokenizer.batch_decode(outputs)
baseline_response = out[0].split("end_header_id|>")[-1].split("<|eot_id|>")[0]
print("--- Baseline raw response ---")
print(baseline_response)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


--- Baseline raw response ---


No, there is no shovel in the scene. The objects in the scene are:

- Two houses (house_1 and house_2)
- A grocery store (grocery_store_1)
- Two sheds (shed_1)

None of these objects are a shovel.


In [13]:
print("--- Baseline raw response ---")
print(baseline_response)

# The base model likely won't produce valid JSON, but try anyway
try:
    baseline_parsed = json.loads(baseline_response)
    print("\n--- Parsed as JSON ---")
    for k, v in baseline_parsed.items():
        print(f"  {k}: {v}")
except json.JSONDecodeError:
    print("\n(Response is not valid JSON — expected for an unfinetuned model)")

JSONDecodeError: Expecting value: line 3 column 1 (char 2)